In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

df = pd.read_csv('../data/cleaned_data.csv')

# --- RFM Segmentation ---
rfm = df.groupby('customer_id').agg(
    recency=('order_date', lambda x: (pd.to_datetime(df['order_date']).max() - pd.to_datetime(x).max()).days),
    frequency=('order_id', 'count'),
    monetary=('revenue', 'sum')
).reset_index()

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['recency','frequency','monetary']])
rfm['segment'] = KMeans(n_clusters=3, random_state=42).fit_predict(rfm_scaled)
rfm['segment_label'] = rfm['segment'].map({0:'At-Risk', 1:'Champions', 2:'Loyal'})
print(rfm['segment_label'].value_counts())

# --- Sales Forecast ---
monthly = df.groupby('month')['revenue'].sum().reset_index()
monthly.columns = ['month','revenue']
X = monthly[['month']]
y = monthly['revenue']
model = LinearRegression().fit(X, y)
future = pd.DataFrame({'month': range(13, 19)})
forecast = model.predict(future)
print("6-month forecast:", forecast.round(0))